# 전통시장 × 시군구 인구: 원자료 → 클리닝 → 검증

**질문** 시군구별 인구 10만 명당 전통시장 수는 얼마인가?

| 자료 | 받은 곳 | 받은 방법 | 기준시점 |
|---|---|---|---|
| 전국전통시장표준데이터 | 공공데이터포털(data.go.kr) | 파일데이터(CSV) 다운로드 | 2025-11-10 |
| 행정구역(시군구)별 성별 인구수 (DT_1B040A3) | KOSIS | 통계청 MCP로 조회 후 저장 | 2025.10 (시장 기준일 직전 월말) |

**규칙** 원자료(`data/raw/`)는 읽기만 한다 · 판정은 출력된 숫자로 한다 · 표본을 바꾸면 바로 다음 줄에서 행 수를 기록한다.

In [1]:
import sys
import pandas as pd
try:
    import dck                  # data-check-kit 점검 함수 (같은 폴더의 dck.py)
except ModuleNotFoundError:     # 저장소에서 열었을 때
    sys.path.append("../../skills/data-check-kit/scripts")
    import dck
pd.set_option("display.max_columns", 30, "display.width", 160)

RAW_MKT = "data/raw/전국전통시장표준데이터.csv"
RAW_POP = "data/raw/KOSIS_시군구별_총인구_202510.csv"
flow = dck.Flow()

## 1단계 · 로딩 직후 점검 — 시장 자료

In [2]:
mkt_raw = dck.read_csv(RAW_MKT, dtype=str)          # 모든 열을 문자로 읽는다(코드·번호의 0 보존)
print("파일 해시:", dck.file_hash(RAW_MKT))
dck.profile(mkt_raw, "시장")

[read] 전국전통시장표준데이터.csv  encoding=cp949  rows=1,393  cols=18
파일 해시: 959aabf13005
== profile: 시장 ==
rows=1,393  cols=18  완전중복행=0


,dtype,결측수,결측률%,고유값수,예시,경고
시장명,object,0,0.00,1343,성정시장,
시장유형,object,0,0.00,7,상설장,
소재지도로명주소,object,14,1.01,1369,충청남도 천안시 서북구 양지21길 31,
소재지지번주소,object,9,0.65,1368,충청남도 천안시 서북구 성정동 736-7,
시장개설주기,object,0,0.00,8,매일,
위도,object,16,1.15,1364,36.81267105,
경도,object,16,1.15,1364,127.1392027,
점포수,object,5,0.36,334,66,
취급품목,object,0,0.00,189,농산물+축산물+가공식품+의류+신발+가정용품+음식점,
사용가능상품권,object,1393,100.00,0,None,


**읽을 것**
- 인코딩이 `cp949`로 잡혔다. 공공데이터포털 CSV는 아직 cp949가 많다. 그냥 `pd.read_csv()`를 쓰면 여기서 오류가 난다.
- `시도`나 `시군구` 열이 없다. 인구 자료와 붙이려면 **주소에서 뽑아내야 한다.**
- `점포수` 결측 5건, `사용가능상품권`은 100% 결측(빈 열)이다.

In [3]:
# 키 후보: 시장명이 한 시장을 가리키는가?
dck.key_check(mkt_raw, "시장명", "시장(시장명)")
mkt_raw["시장명"].value_counts().head(5)

[key] 시장(시장명) ['시장명']: 유일하지 않음  결측키=0  중복키 관측치=79
       시장명 시장유형                   소재지도로명주소                 소재지지번주소
946   강남시장  상설장    경기도 부천시 원미구 옥산로154번길 21  경기도 부천시 원미구 도당동 274-54
566   강남시장  상설장       서울특별시 강남구 압구정로 2길 46    서울특별시 강남구 신사동 510-11
1247  교동시장  상설장       전라남도 여수시 교동시장1길 17-6       전라남도 여수시 교동 605-3
924   교동시장  상설장     대구광역시 중구 교동길 40 (교동시장)       대구광역시 중구 교동 67-12
849   구암시장  상설장        대구광역시 북구 태암남로11길 18        대구광역시 북구 구암동 693
1113  구암시장  상설장  경상남도 창원시 마산 회원구 구암남 14길 8   경상남도 창원시 마산회원구 구암동 87
1307  남부시장  상설장          경상북도 포항시 남구 상공로43     경상북도 포항시 남구 상도동 629
839   남부시장  상설장         대구광역시 남구 현충로 19길43      대구광역시 남구 대명동 797-1
828   대곡시장  상설장            대구광역시 달서구 도원로24      대구광역시 달서구 도원동 1432
1080  대곡시장  상설장  경상남도 진주시 대곡면 진의로 1079번길 6  경상남도 진주시 대곡면 광석리 321-7


시장명
중앙시장     12
역전시장      5
신중앙시장     4
동부시장      4
자유시장      3
Name: count, dtype: int64

In [4]:
dck.key_check(mkt_raw, ["시장명", "소재지도로명주소"], "시장(시장명+도로명주소)");

[key] 시장(시장명+도로명주소) ['시장명', '소재지도로명주소']: 유일하지 않음  결측키=14  중복키 관측치=0


도로명주소가 빈 행이 14개라 결측키가 생긴다. 빈 도로명주소는 지번주소로 채워 키를 만든다.

In [5]:
mkt_raw_key = mkt_raw.assign(주소=mkt_raw["소재지도로명주소"].fillna(mkt_raw["소재지지번주소"]))
dck.key_check(mkt_raw_key, ["시장명", "주소"], "시장(시장명+주소)");

[key] 시장(시장명+주소) ['시장명', '주소']: 유일  결측키=0  중복키 관측치=0


'중앙시장'만 12곳이다. **시장명은 키가 아니다.** 시장명+주소는 유일하다.
→ 이것은 '중복 시장을 지워야 한다'는 뜻이 아니라, 이름이 같은 다른 시장이 있다는 뜻이다. 지우면 안 된다.

In [6]:
flow.start(mkt_raw, "시장 원자료", "data.go.kr 전국전통시장표준데이터, 기준일 2025-11-10");

[flow] 시작 시장 원자료: 1,393행


## 2단계 · 코드체계·명칭 점검 — 주소에서 시도·시군구 뽑기

In [7]:
addr = mkt_raw["소재지도로명주소"].fillna(mkt_raw["소재지지번주소"]).str.strip()
tok = addr.str.split()
tok.str[0].value_counts()

소재지도로명주소
서울특별시      195
경상남도       159
부산광역시      155
경기도        149
경상북도       137
대구광역시      106
전라남도        96
강원특별자치도     61
전북특별차치도     57
충청남도        56
충청북도        56
인천광역시       45
울산광역시       42
대전광역시       27
광주광역시       25
제주특별자치도     23
세종특별자치시      4
Name: count, dtype: int64

**찾았다:** `전북특별차치도`(오탈자, '자치' → '차치')가 57행이다. 이대로 붙이면 전북의 57개 시장이 통째로 미매칭이 된다.

In [8]:
# 주소 두 번째 토큰의 끝 글자 — 시·군·구가 아니면 무엇인가?
second = tok.str[1]
pd.DataFrame({"시도": tok.str[0], "둘째": second})[~second.str[-1].isin(list("시군구"))]

,시도,둘째
359,세종특별자치시,전의면
744,세종특별자치시,조치원읍
746,세종특별자치시,부강면
747,세종특별자치시,금남면


세종특별자치시는 아래에 시군구가 없어서 둘째 토큰이 읍·면이다. 인구 자료에서는 `세종시` 한 줄이다.

In [9]:
# 도로명주소와 지번주소의 시군구가 서로 다른 행
t2 = mkt_raw["소재지지번주소"].str.split()
both = mkt_raw["소재지도로명주소"].notna() & mkt_raw["소재지지번주소"].notna()
conflict = both & ((tok.str[0] + tok.str[1]) != (t2.str[0] + t2.str[1]))
mkt_raw.loc[conflict, ["시장명", "소재지도로명주소", "소재지지번주소"]]

,시장명,소재지도로명주소,소재지지번주소
430,용남시장,전북특별차치도 남원시 동헌길10,전북특별차치도 고창군 흥덕면 흥덕리 430-1


한 시장의 도로명주소는 남원시, 지번주소는 고창군이다. 어느 쪽이 맞는지는 **코드로 판정할 수 없다.**
규칙: 도로명주소를 우선하고, 이 행을 `확인 필요` 목록에 남긴다.

## 3단계 · 결합 규칙 (머지 전에 적는다)

| 항목 | 규칙 |
|---|---|
| 분석단위 | 시군구 (도의 일반구는 상위 시로 묶음, 광역시의 자치구는 그대로) |
| 결합 키 | **시도 + 시군구** (시군구명만 쓰면 '중구'·'고성군'·'광주시'가 겹친다) |
| 키 표준화 | 시도 오탈자 교정(`전북특별차치도`→`전북특별자치도`), 세종 → `세종시` |
| 결합 방향 | 시장(왼쪽) ← 인구(오른쪽), left |
| 허용 관계 | **N:1** (한 시군구에 여러 시장) · **금지 N:M** |
| 인구 자료 제외 | 전국·시도 합계 행, 일반구 행(상위 시와 이중 집계), 출장소 행(인구 0) |
| 예상 결과 | 시장 1,393행 그대로, 시장 쪽 미매칭 0 |
| 허용 미매칭 | 시장 쪽 0건. 인구 쪽 미매칭은 '전통시장이 없는 시군구'로 해석한다 |

## 4단계 · 클리닝 — 단계마다 기록

In [10]:
mkt = mkt_raw.copy()
mkt["시도"] = tok.str[0].replace({"전북특별차치도": "전북특별자치도"})
mkt["시군구"] = tok.str[1].where(mkt["시도"] != "세종특별자치시", "세종시")
mkt["점포수"] = pd.to_numeric(mkt["점포수"], errors="coerce")
mkt["개설연도"] = pd.to_numeric(mkt["개설연도"], errors="coerce")
mkt["주소충돌"] = conflict

print("시도 교정된 셀:", (tok.str[0] != mkt["시도"]).sum())
print("세종 교정된 셀:", (tok.str[1] != mkt["시군구"]).sum())
mkt = flow.step(mkt_raw, mkt, "시도·시군구 생성", "값 변환만, 행 수 불변", expect=len(mkt_raw))

시도 교정된 셀: 57
세종 교정된 셀: 4
[flow] 시도·시군구 생성: 1,393 → 1,393 (+0)  기대값 일치


### 인구 자료 정리

In [11]:
pop_try = dck.read_csv(RAW_POP)          # 그냥 읽으면?
pop_try.iloc[:, 1].unique()

[read] KOSIS_시군구별_총인구_202510.csv  encoding=utf-8  rows=293  cols=3


array([2025.1])

**함정:** 파일에는 `2025.10`(10월)이라고 적혀 있는데 숫자로 읽히면서 `2025.1`이 됐다. 1월처럼 보인다.
날짜·코드·번호 열은 문자로 읽는다.

In [12]:
pop_raw = dck.read_csv(RAW_POP, dtype={"시점": str})
pop_raw.columns = ["행정구역", "시점", "인구"]
print(pop_raw["시점"].unique())
dck.profile(pop_raw, "인구")
dck.key_check(pop_raw, "행정구역", "인구(행정구역명)");

[read] KOSIS_시군구별_총인구_202510.csv  encoding=utf-8  rows=293  cols=3
['2025.10']
== profile: 인구 ==
rows=293  cols=3  완전중복행=0
[key] 인구(행정구역명) ['행정구역']: 유일하지 않음  결측키=0  중복키 관측치=31
    행정구역       시점      인구
17   강서구  2025.10  550248
39   강서구  2025.10  148922
274  고성군  2025.10   26800
247  고성군  2025.10   47150
71    남구  2025.10  207912
204   남구  2025.10  216338
34    남구  2025.10  251227
82    남구  2025.10  304512
48    남구  2025.10  135692
69    동구  2025.10  105148


행정구역명만으로는 유일하지 않다('중구' 6개 등). 이 표에는 **시도 열이 없고**, 시도 행 아래로 하위 행이 이어지는 계층 구조다.

In [13]:
SIDO = ["서울특별시","부산광역시","대구광역시","인천광역시","광주광역시","대전광역시","울산광역시",
        "세종특별자치시","경기도","충청북도","충청남도","전라남도","경상북도","경상남도",
        "제주특별자치도","강원특별자치도","전북특별자치도"]
pop = pop_raw.copy()
pop["시도"] = pop["행정구역"].where(pop["행정구역"].isin(SIDO)).ffill()
metro = pop["시도"].fillna("").str.endswith(("특별시", "광역시", "특별자치시"))
pop["수준"] = "시군구"
pop.loc[pop["행정구역"] == "전국", "수준"] = "전국"
pop.loc[pop["행정구역"].isin(SIDO), "수준"] = "시도"
pop.loc[pop["행정구역"].str.contains("출장소"), "수준"] = "출장소"
pop.loc[~metro & pop["행정구역"].str.endswith("구") & (pop["수준"] == "시군구"), "수준"] = "일반구"
pop["수준"].value_counts()

수준
시군구    229
일반구     35
시도      17
출장소     11
전국       1
Name: count, dtype: int64

In [14]:
# 검증: 시도마다 시군구 합 = 시도 합계? (일반구를 빼야 맞아야 한다)
part = pop[pop["수준"] == "시군구"].groupby("시도")["인구"].sum()
total = pop[pop["수준"] == "시도"].set_index("행정구역")["인구"]
gap = (part - total)
print("시도 17개 중 불일치:", (gap != 0).sum())
dck.total_check(pop[pop["수준"] == "시도"], "인구", pop.loc[pop["행정구역"] == "전국", "인구"].item(), "시도 합 vs 전국");

시도 17개 중 불일치: 0
[total] 시도 합 vs 전국: 계산 51,143,421 vs 공표 51,143,421  차이 +0.0000%  일치


In [15]:
# 일반구를 빼지 않았다면?
wrong = pop[pop["수준"].isin(["시군구", "일반구"])]["인구"].sum()
print(f"일반구 포함 합계 {wrong:,} — 전국보다 {wrong - 51143421:,}명 많다(이중 집계)")

일반구 포함 합계 60,960,025 — 전국보다 9,816,604명 많다(이중 집계)


In [16]:
pop_sgg = pop[pop["수준"] == "시군구"].rename(columns={"행정구역": "시군구"})[["시도", "시군구", "인구"]]
pop_sgg = flow.step(pop_raw, pop_sgg, "인구: 시군구만 남김", "전국·시도·일반구·출장소 행 제외", expect=229)
dck.key_check(pop_sgg, ["시도", "시군구"], "인구(시도+시군구)");

[flow] 인구: 시군구만 남김: 293 → 229 (-64)  기대값 일치
[key] 인구(시도+시군구) ['시도', '시군구']: 유일  결측키=0  중복키 관측치=0


## 5단계 · 결합 회계

### ① 잘못된 키로 붙이면 (시군구명만)

In [17]:
_ = dck.merge_audit(mkt, pop_sgg, on="시군구", names=("시장", "인구"), show=5)

== merge_audit 시장 LEFT JOIN 인구 on ['시군구'] ==
  시장_rows: 1,393
  인구_rows: 229
  merged_rows: 2,402
  expected_rows: 2,402
  회계식: 일치
  matched_rows: 2,402
  시장_only_rows: 0
  인구_only_rows: 4
  시장_매칭률%: 100.0
  인구_매칭률%: 98.25
  키_중복_증폭: 1.7243
  관계_1:1: 22
  관계_N:1: 174
  관계_1:N: 0
  관계_N:M: 7
  [!] N:M 키가 있다. 결합 규칙에서 금지했다면 여기서 멈춘다.
  미매칭 키 4개 (앞 4개)
시군구    쪽
옹진군 인구에만
양주시 인구에만
계룡시 인구에만
울릉군 인구에만


- 시장은 1,393개인데 결합 결과가 더 많다. **키 중복 증폭 > 1**이면 시장 행이 복제된 것이다.
- N:M 관계가 생겼다. 서울 중구 시장에 부산·대구·인천 중구 인구까지 붙었다.
- 에러는 나지 않는다. 숫자를 보지 않으면 모르고 지나간다.

### ② 시도 오탈자를 고치지 않았다면

In [18]:
mkt_typo = mkt.assign(시도=tok.str[0])
_, rep_typo, um_typo = dck.merge_audit(mkt_typo, pop_sgg, on=["시도", "시군구"], names=("시장", "인구"), show=3)

== merge_audit 시장 LEFT JOIN 인구 on ['시도', '시군구'] ==
  시장_rows: 1,393
  인구_rows: 229
  merged_rows: 1,393
  expected_rows: 1,393
  회계식: 일치
  matched_rows: 1,336
  시장_only_rows: 57
  인구_only_rows: 18
  시장_매칭률%: 95.91
  인구_매칭률%: 92.14
  키_중복_증폭: 1.0
  관계_1:1: 22
  관계_N:1: 189
  관계_1:N: 0
  관계_N:M: 0
  미매칭 키 32개 (앞 3개)
     시도 시군구    쪽
전북특별차치도 전주시 시장에만
전북특별차치도 익산시 시장에만
전북특별차치도 완주군 시장에만


### ③ 규칙대로 (시도+시군구, N:1 강제)

In [19]:
merged, rep, unmatched = dck.merge_audit(mkt, pop_sgg, on=["시도", "시군구"], how="left",
                                         validate="many_to_one", names=("시장", "인구"), show=10)
merged = flow.step(mkt, merged.drop(columns="_merge"), "인구 결합", "시도+시군구, N:1", expect=len(mkt))

== merge_audit 시장 LEFT JOIN 인구 on ['시도', '시군구'] ==
  시장_rows: 1,393
  인구_rows: 229
  merged_rows: 1,393
  expected_rows: 1,393
  회계식: 일치
  matched_rows: 1,393
  시장_only_rows: 0
  인구_only_rows: 4
  시장_매칭률%: 100.0
  인구_매칭률%: 98.25
  키_중복_증폭: 1.0
  관계_1:1: 23
  관계_N:1: 202
  관계_1:N: 0
  관계_N:M: 0
  미매칭 키 4개 (앞 4개)
   시도 시군구    쪽
인천광역시 옹진군 인구에만
  경기도 양주시 인구에만
 충청남도 계룡시 인구에만
 경상북도 울릉군 인구에만
[flow] 인구 결합: 1,393 → 1,393 (+0)  기대값 일치


In [20]:
# 미매칭 목록은 한 줄씩 읽는다: 인구에만 있는 시군구 = 전통시장이 없는 곳인가?
unmatched["쪽"].value_counts()

쪽
인구에만    4
Name: count, dtype: int64

In [21]:
unmatched[unmatched["쪽"] == "인구에만"].head(20)

,시도,시군구,쪽
0,인천광역시,옹진군,인구에만
1,경기도,양주시,인구에만
2,충청남도,계룡시,인구에만
3,경상북도,울릉군,인구에만


시장 쪽 미매칭은 0이다(규칙 충족). 인구에만 있는 4개 시군구는 이 표준데이터에 전통시장이 한 곳도 없는 곳이다.
→ 이것은 '이 자료에 등록된 시장이 없다'는 뜻이지, '실제로 시장이 없다'는 뜻은 아니다. 분석에서 0으로 넣을지는 7단계 질문에 답한 뒤 정한다.

## 6단계 · 분포·외부 대조

In [22]:
dck.truncation_flags(dck.dist(merged, ["점포수", "개설연도"]))

,변수,N,결측률%,평균,표준편차,min,p1,p5,p25,p50,p75,p95,p99,max,0비율%,최댓값_동일건수,점검신호
0,점포수,1388,0.36,143.309798,280.007177,4.0,9.00,21.0,51.0,82.0,140.0,406.0,1208.29,5111.0,0.0,1,
1,개설연도,1380,0.93,1977.434783,24.381562,1780.0,1911.79,1929.0,1965.0,1978.0,1994.0,2013.0,2018.00,2023.0,0.0,2,p99가 최댓값에 바짝 붙음


`점검신호`는 판정이 아니라 들여다볼 곳 표시다. 개설연도의 'p99가 최댓값에 바짝 붙음'은 연도 변수라 당연한 결과다(최근 연도가 상한).
점포수는 중앙값 82, 최댓값 5,111로 꼬리가 길다. 평균(143)보다 중앙값을 보고하는 편이 낫다.

In [23]:
merged.nsmallest(5, "개설연도")[["시장명", "시도", "시군구", "개설연도"]]

,시장명,시도,시군구,개설연도
1109,마산어시장,경상남도,창원시,1780.0
1079,진주중앙시장,경상남도,진주시,1884.0
1220,성주시장,경상북도,성주군,1887.0
433,금만시장,전북특별자치도,김제시,1890.0
434,김제전통시장,전북특별자치도,김제시,1900.0


개설연도 1780년은 오류일까, 실제로 오래된 시장일까? 이 표만으로는 알 수 없다. 값만 보고 지우거나 고치지 않는다. **근거(지자체·시장 자료)를 확인하기 전까지는 `확인 필요`로 둔다.**

In [24]:
# 외부 대조: 정리된 인구 229개 시군구 합 = 전국 인구?
dck.total_check(pop_sgg, "인구", 51143421, "시군구 합 vs 전국");

[total] 시군구 합 vs 전국: 계산 51,143,421 vs 공표 51,143,421  차이 +0.0000%  일치


## 분석 자료 만들기 — 시군구 단위

In [25]:
sgg = (merged.groupby(["시도", "시군구"])
             .agg(시장수=("시장명", "size"), 점포수=("점포수", "sum"))
             .reset_index())
out = pop_sgg.merge(sgg, on=["시도", "시군구"], how="left", validate="one_to_one")
out[["시장수", "점포수"]] = out[["시장수", "점포수"]].fillna(0).astype(int)
out["인구10만명당_시장수"] = (out["시장수"] / out["인구"] * 1e5).round(2)
out = flow.step(pop_sgg, out, "시군구 단위 분석자료", "시장 없는 시군구는 0", expect=229)
print("시장수 합:", out["시장수"].sum(), "(원자료 1,393과 같아야 한다)")
out.sort_values("인구10만명당_시장수", ascending=False).head(10)

[flow] 시군구 단위 분석자료: 229 → 229 (+0)  기대값 일치
시장수 합: 1393 (원자료 1,393과 같아야 한다)


,시도,시군구,인구,시장수,점포수,인구10만명당_시장수
25,부산광역시,중구,36645,16,3167,43.66
166,경상북도,청송군,23424,6,165,25.61
1,서울특별시,중구,118331,25,20490,21.13
144,전라남도,장흥군,34205,6,240,17.54
223,전북특별자치도,무주군,22813,4,101,17.53
190,경상남도,하동군,40106,7,375,17.45
110,충청북도,보은군,30256,5,328,16.53
142,전라남도,보성군,36660,6,309,16.37
126,충청남도,금산군,49014,8,926,16.32
175,경상북도,울진군,45930,7,389,15.24


1위가 부산 중구다. 시장이 많아서라기보다 **분모(상주인구)가 작아서**다. 도심 구는 낮 인구와 주민등록 인구의 차이가 크다.
→ 지표를 고를 때 분모가 무엇을 재는지가 결과를 바꾼다. 이것은 클리닝이 아니라 연구자가 판단할 몫이다.

## 7단계 · 의미 점검 (사람이 답한다)

| 질문 | 답 | 근거 |
|---|---|---|
| 표준데이터의 시장은 전수인가? 등록 시장만인가? | 확인 필요 | 데이터 설명서 |
| 점포수는 빈 점포를 포함하는가? | 확인 필요 | 제공기관 문의 |
| 인구는 주민등록인구다. 외국인은 포함되는가? | 미포함 | KOSIS 통계설명자료 |
| 인구 기준시점(2025.10)과 시장 기준일(2025-11-10)이 맞는가? | 직전 월말로 맞춤 | 3단계 규칙 |
| 주소가 서로 다른 1개 시장은 어디에 넣었나? | 도로명주소 기준 | 4단계 |

## 8단계 · 표본 흐름표와 저장

In [26]:
ft = flow.table()
ft

,단계,전,후,변화,사유
0,시장 원자료,<NA>,1393,<NA>,"data.go.kr 전국전통시장표준데이터, 기준일 2025-11-10"
1,시도·시군구 생성,1393,1393,0,"값 변환만, 행 수 불변"
2,인구: 시군구만 남김,293,229,-64,전국·시도·일반구·출장소 행 제외
3,인구 결합,1393,1393,0,"시도+시군구, N:1"
4,시군구 단위 분석자료,229,229,0,시장 없는 시군구는 0


In [27]:
import os
os.makedirs("data/clean", exist_ok=True)
merged.to_csv("data/clean/market_with_pop.csv", index=False, encoding="utf-8-sig")
out.to_csv("data/clean/sgg_market_per_pop.csv", index=False, encoding="utf-8-sig")
mkt.loc[mkt["주소충돌"], ["시장명", "소재지도로명주소", "소재지지번주소"]].to_csv(
    "data/clean/check_needed.csv", index=False, encoding="utf-8-sig")
ft.to_csv("data/clean/sample_flow.csv", index=False, encoding="utf-8-sig")
print(sorted(os.listdir("data/clean")))

['check_needed.csv', 'market_with_pop.csv', 'sample_flow.csv', 'sgg_market_per_pop.csv']


### 다음: 독립 감사
이 노트북을 만든 AI와 **다른 AI**에게 숫자만으로 다시 확인시킨다.

> `data-auditor` 에이전트로 감사해줘. 원자료 `data/raw/`, 정제본 `data/clean/`, 결합 규칙은 노트북 3단계, 흐름표 `data/clean/sample_flow.csv`.